## Final project (1_Data_cleaning), Irina Zasenko

## 1. Очистка и подготовка данных


In [1]:
# импорт библиотек
!pip install pandas numpy
!pip install openpyxl
import pandas as pd
import numpy as np
import pickle
from datetime import time, timedelta

### 1.1. Анализ датасета Calls

In [2]:
# Загрузим датасет Calls
#calls = pd.read_excel('Calls.xlsx')
calls = pd.read_excel("Calls.xlsx", dtype={"Id": str, "CONTACTID": str})
# Просмотрим первые строки таблицы, чтобы понять структуру данных
calls.head()

,Id,Call Start Time,Call Owner Name,CONTACTID,Call Type,Call Duration (in seconds),Call Status,Dialled Number,Outgoing Call Status,Scheduled in CRM,Tag
0,5805028000000805001,30.06.2023 08:43,John Doe,NaN,Inbound,171.0,Received,NaN,NaN,NaN,NaN
1,5805028000000768006,30.06.2023 08:46,John Doe,NaN,Outbound,28.0,Attended Dialled,NaN,Completed,0.0,NaN
2,5805028000000764027,30.06.2023 08:59,John Doe,NaN,Outbound,24.0,Attended Dialled,NaN,Completed,0.0,NaN
3,5805028000000787003,30.06.2023 09:20,John Doe,5805028000000645014,Outbound,6.0,Attended Dialled,NaN,Completed,0.0,NaN
4,5805028000000768019,30.06.2023 09:30,John Doe,5805028000000645014,Outbound,11.0,Attended Dialled,NaN,Completed,0.0,NaN


In [3]:
# Проверка дубликатов (найдено - 0)
calls.duplicated().sum()

np.int64(0)

In [4]:
# вывод информации о данных и о пропущенных значениях
calls.info()
calls.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 95874 entries, 0 to 95873
Data columns (total 11 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   Id                          95874 non-null  object 
 1   Call Start Time             95874 non-null  object 
 2   Call Owner Name             95874 non-null  object 
 3   CONTACTID                   91941 non-null  object 
 4   Call Type                   95874 non-null  object 
 5   Call Duration (in seconds)  95791 non-null  float64
 6   Call Status                 95874 non-null  object 
 7   Dialled Number              0 non-null      float64
 8   Outgoing Call Status        86875 non-null  object 
 9   Scheduled in CRM            86875 non-null  float64
 10  Tag                         0 non-null      float64
dtypes: float64(4), object(7)
memory usage: 8.0+ MB


Id                                0
Call Start Time                   0
Call Owner Name                   0
CONTACTID                      3933
Call Type                         0
Call Duration (in seconds)       83
Call Status                       0
Dialled Number                95874
Outgoing Call Status           8999
Scheduled in CRM               8999
Tag                           95874
dtype: int64

a. Id: Уникальный идентификатор для каждого звонка: уже при загрузке был преобразован в строку str

b. Call Start Time - Время начала звонка : object - преобразовать в datetime для дальнейшего анализа временных рядов

c. Call Owner Name - Имя лица, ответственного за звонок: object - преобразовать в категориальный тип данных

d. CONTACTID - Уникальный идентификатор контакта:  уже при загрузке был преобразован в строку str

e. Call Type - Тип звонка: object - преобразовать в категориальный тип данных

f. Call Duration (in seconds) - Длительность звонка в секундах: float64 - преобразовать в int32, т.к. это целые числа

g. Call Status - Окончательный статус звонка: object - преобразовать в категориальный тип данных

h. Dialled Number: Набранный номер телефона

i. Outgoing Call Status - Статус исходящих вызовов: object - преобразовать в категориальный тип данных

j. Scheduled in CRM - Указывает, был ли звонок запланирован через систему CRM: float64 преобразовать в категориальный тип данных, чтобы эффективнее хранить и обрабатывать значения, так как они представляют собой бинарные состояния (запланировано или нет)

k. Tag: Тэг вызова.


Анализ и предложения по пропущенным значениям:

Call Duration (in seconds) пропущенные значения были заполнены медианным значением. В этом случае использование медианы позволит сохранить более точное представление о типичной длительности звонков, поэтому выбрана медиана вместо среднего.

Dialled Number - удалить столбец, т.к. все значения 0 non-null

Outgoing Call Status пропущенные значения были заполнены значением "Unknown". Значение "Unknown" явно обозначает, что статус исходящего звонка не был указан. Это лучше, чем удалять строки или оставлять пропуски, так как сохраняется структура данных для последующего анализа.

Scheduled in CRM пропущенные значения были заполнены значением "Unknown". Значения "Unknown" указывает на то, что мы не знаем, был ли звонок запланирован в CRM или не был.

CONTACTID пропущенные значения удалены. В данном случае, наиболее логичным решением будет удаление строк с пропусками, так как это идентификатор, и он должен быть заполнен для корректного соединения с другими таблицами.

Tag - удалить столбец, т.к. все значения 0 non-null

In [5]:
# Преобразование типов данных и обработка пропущенных значений

calls['Call Start Time'] = pd.to_datetime(calls['Call Start Time'], format='%d.%m.%Y %H:%M', dayfirst=True, errors='coerce')

calls = calls.dropna(subset=['CONTACTID']) # cначала удаляем, потом изменяем тип данных
calls['Id'] = calls['Id'].astype(str)
calls['Call Duration (in seconds)'] = calls['Call Duration (in seconds)'].fillna(calls['Call Duration (in seconds)'].median())
calls['Call Duration (in seconds)'] = calls['Call Duration (in seconds)'].astype('int32')

In [6]:
# Создадим переменную: список столбцов которые нужно преобразовать в категориальный тип данных
category_columns = ['Call Owner Name', 'Call Type', 'Call Status', 'Outgoing Call Status', 'Scheduled in CRM']

In [7]:
# Преобразуем нужные столбцы
for col in category_columns:
    calls[col] = calls[col].astype('category')

In [8]:
# Заполним пропущенные значения и добавим категории 'Unknown'
for col in ['Outgoing Call Status', 'Scheduled in CRM']:
    if 'Unknown' not in calls[col].cat.categories:
        calls[col] = calls[col].cat.add_categories(['Unknown'])
    calls[col] = calls[col].fillna('Unknown')

In [9]:
# удаление ненужных столбцов
calls = calls.drop(columns=['Dialled Number', 'Tag'])

In [10]:
calls = calls.rename(columns={
    'Id': 'call_id',
    'Call Start Time': 'call_started_at',
    'Call Owner Name': 'manager',
    'CONTACTID': 'contact_id',
    'Call Duration (in seconds)': 'call_duration_sec'
})

In [11]:
# Проверка данных после заполнения пропусков и преобразования
print(calls.isnull().sum(), "\n")
calls.info()
calls.head()

call_id                 0
call_started_at         0
manager                 0
contact_id              0
Call Type               0
call_duration_sec       0
Call Status             0
Outgoing Call Status    0
Scheduled in CRM        0
dtype: int64 

<class 'pandas.core.frame.DataFrame'>
Index: 91941 entries, 3 to 95872
Data columns (total 9 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   call_id               91941 non-null  object        
 1   call_started_at       91941 non-null  datetime64[ns]
 2   manager               91941 non-null  category      
 3   contact_id            91941 non-null  object        
 4   Call Type             91941 non-null  category      
 5   call_duration_sec     91941 non-null  int32         
 6   Call Status           91941 non-null  category      
 7   Outgoing Call Status  91941 non-null  category      
 8   Scheduled in CRM      91941 non-null  category      
dtypes: ca

,call_id,call_started_at,manager,contact_id,Call Type,call_duration_sec,Call Status,Outgoing Call Status,Scheduled in CRM
3,5805028000000787003,2023-06-30 09:20:00,John Doe,5805028000000645014,Outbound,6,Attended Dialled,Completed,0.0
4,5805028000000768019,2023-06-30 09:30:00,John Doe,5805028000000645014,Outbound,11,Attended Dialled,Completed,0.0
5,5805028000000790004,2023-06-30 12:09:00,John Doe,5805028000000645014,Outbound,12,Attended Dialled,Completed,0.0
6,5805028000000773022,2023-06-30 14:24:00,John Doe,5805028000000645014,Outbound,4,Attended Dialled,Completed,0.0
10,5805028000000942048,2023-07-04 15:35:00,Jane Smith,5805028000000645014,Outbound,20,Attended Dialled,Completed,0.0


### 1.2. Анализ датасета Contacts

In [12]:
# Загрузим датасет Contacts
contacts = pd.read_excel('Contacts.xlsx', dtype={"Id": str})
# Просмотрим первые строки таблицы, чтобы понять структуру данных
contacts.head()

,Id,Contact Owner Name,Created Time,Modified Time
0,5805028000000645014,Rachel White,27.06.2023 11:28,22.12.2023 13:34
1,5805028000000872003,Charlie Davis,03.07.2023 11:31,21.05.2024 10:23
2,5805028000000889001,Bob Brown,02.07.2023 22:37,21.12.2023 13:17
3,5805028000000907006,Bob Brown,03.07.2023 05:44,29.12.2023 15:20
4,5805028000000939010,Nina Scott,04.07.2023 10:11,16.04.2024 16:14


In [13]:
# Проверка дубликатов (найдено - 0)
contacts.duplicated().sum()

np.int64(0)

In [14]:
# вывод информации о данных и о пропущенных значениях
contacts.info()
contacts.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18548 entries, 0 to 18547
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   Id                  18548 non-null  object
 1   Contact Owner Name  18548 non-null  object
 2   Created Time        18548 non-null  object
 3   Modified Time       18548 non-null  object
dtypes: object(4)
memory usage: 579.8+ KB


Id                    0
Contact Owner Name    0
Created Time          0
Modified Time         0
dtype: int64

Анализ и предложения по типам данных:

ID:  уже при загрузке был преобразован в строку

Created Time: object - преобразовать в datetime для дальнейшего анализа временных рядов

Modified Time: object - преобразовать в datetime для дальнейшего анализа временных рядов

Contact Owner Name: object - преобразовать в категориальный тип данных.

In [15]:
contacts['Created Time'] = pd.to_datetime(contacts['Created Time'], format='%d.%m.%Y %H:%M', dayfirst=True, errors='coerce')
contacts['Modified Time'] = pd.to_datetime(contacts['Modified Time'], format='%d.%m.%Y %H:%M', dayfirst=True, errors='coerce')
contacts['Contact Owner Name'] = contacts['Contact Owner Name'].astype('category')

In [16]:
contacts = contacts.rename(columns={
    'Id': 'contact_id',
    'Contact Owner Name': 'contact_owner_name',
    'Created Time': 'created_time',
    'Modified Time': 'modified_time'
})

In [17]:
# Проверка данных после преобразования
contacts.info()
contacts.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18548 entries, 0 to 18547
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   contact_id          18548 non-null  object        
 1   contact_owner_name  18548 non-null  category      
 2   created_time        18548 non-null  datetime64[ns]
 3   modified_time       18548 non-null  datetime64[ns]
dtypes: category(1), datetime64[ns](2), object(1)
memory usage: 454.2+ KB


,contact_id,contact_owner_name,created_time,modified_time
0,5805028000000645014,Rachel White,2023-06-27 11:28:00,2023-12-22 13:34:00
1,5805028000000872003,Charlie Davis,2023-07-03 11:31:00,2024-05-21 10:23:00
2,5805028000000889001,Bob Brown,2023-07-02 22:37:00,2023-12-21 13:17:00
3,5805028000000907006,Bob Brown,2023-07-03 05:44:00,2023-12-29 15:20:00
4,5805028000000939010,Nina Scott,2023-07-04 10:11:00,2024-04-16 16:14:00


### 1.3. Анализ датасета Deals

In [18]:
# Загрузим датасет Deals

deals = pd.read_excel("Deals.xlsx", dtype={"Id": str, "Contact Name": str})

# Просмотрим первые строки таблицы, чтобы понять структуру данных
deals.head()

,Id,Deal Owner Name,Closing Date,Quality,Stage,Lost Reason,Page,Campaign,SLA,Content,...,Product,Education Type,Created Time,Course duration,Months of study,Initial Amount Paid,Offer Total Amount,Contact Name,City,Level of Deutsch
0,5805028000056864695,Ben Hall,NaN,NaN,New Lead,NaN,/eng/test,03.07.23women,NaN,v16,...,NaN,NaN,21.06.2024 15:30,NaN,NaN,NaN,NaN,5805028000056849495,NaN,NaN
1,5805028000056859489,Ulysses Adams,NaN,NaN,New Lead,NaN,/at-eng,NaN,NaN,NaN,...,Web Developer,Morning,21.06.2024 15:23,6.0,NaN,0,2000,5805028000056834471,NaN,NaN
2,5805028000056832357,Ulysses Adams,21.06.2024,D - Non Target,Lost,Non target,/at-eng,engwien_AT,00:26:43,b1-at,...,NaN,NaN,21.06.2024 14:45,NaN,NaN,NaN,NaN,5805028000056854421,NaN,NaN
3,5805028000056824246,Eva Kent,21.06.2024,E - Non Qualified,Lost,Invalid number,/eng,04.07.23recentlymoved_DE,01:00:04,bloggersvideo14com,...,NaN,NaN,21.06.2024 13:32,NaN,NaN,NaN,NaN,5805028000056889351,NaN,NaN
4,5805028000056873292,Ben Hall,21.06.2024,D - Non Target,Lost,Non target,/eng,discovery_DE,00:53:12,website,...,NaN,NaN,21.06.2024 13:21,NaN,NaN,NaN,NaN,5805028000056876176,NaN,NaN


In [19]:
# Проверка дубликатов (найдено - 3)
deals.duplicated().sum()

np.int64(0)

In [20]:
# Удаление дубликатов (удалено - 3)
deals = deals.drop_duplicates()

In [21]:
# вывод информации о данных
deals.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21595 entries, 0 to 21594
Data columns (total 23 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Id                   21593 non-null  object 
 1   Deal Owner Name      21564 non-null  object 
 2   Closing Date         14645 non-null  object 
 3   Quality              19340 non-null  object 
 4   Stage                21593 non-null  object 
 5   Lost Reason          16124 non-null  object 
 6   Page                 21593 non-null  object 
 7   Campaign             16067 non-null  object 
 8   SLA                  15533 non-null  object 
 9   Content              14147 non-null  object 
 10  Term                 12454 non-null  object 
 11  Source               21593 non-null  object 
 12  Payment Type         496 non-null    object 
 13  Product              3592 non-null   object 
 14  Education Type       3300 non-null   object 
 15  Created Time         21593 non-null 

### Анализ и предложения по типам данных:*

Id: float64 - преобразован при загрузке в str, так как это идентификатор.

Deal Owner Name: object - преобразовать в категориальный тип данных.

Closing Date: object - преобразовать в datetime для дальнейшего анализа временных рядов.

Quality: object - преобразовать в категориальный тип данных.

Stage: object - преобразовать в категориальный тип данных.

Lost Reason: object - преобразовать в категориальный тип данных.

Page: object - преобразовать в категориальный тип данных.

Campaign: object - преобразовать в категориальный тип данных.

SLA: object - преобразуем значения в этом столбце в секунды (преобразуем с помощью функции).

Content: object - преобразовать в категориальный тип данных.

Term: object - преобразовать в категориальный тип данных.

Source: object - преобразовать в категориальный тип данных.

Payment Type: object - преобразовать в категориальный тип данных.

Product: object - преобразовать в категориальный тип данных.

Education Type: object - преобразовать в категориальный тип данных.

Created Time: object -преобразовать в datetime для дальнейшего анализа временных рядов.

Course duration: float64 - преобразовать в Int8.

Months of study: float64 - преобразовать в Int8.

Initial Amount Paid: object - преобразовать в Int32 для анализа финансовых данных (есть нечисловые значения - обработать).

Offer Total Amount: object - преобразовать в Int32 для анализа финансовых данных (есть нечисловые значения - обработать).

Contact Name: float64 - преобразовать при загрузке в str, так как это идентификатор.

City: object - преобразовать в категориальный тип данных.

Level of Deutsch: object - преобразовать в категориальный тип данных.

*P.S. Преобразование столбцов в категориальный тип данных выполненно с целью уменьшения объема памяти и ускорения операций группировки.


In [22]:
# Вывод уникальных значений в столбце перед преобразованием, потому что там содержатся нечисловые данные, такие как '€' и запятые)
print(deals['Initial Amount Paid'].unique())
print(deals['Offer Total Amount'].unique())

[nan 0 1000 '€ 3.500,00' 500 100 4500 300 200 2000 11000 4000 3000 3500
 11500 1200 1500 1 5000 600 700 350 9 400 450]
[nan 2000 9000 11000 3500 4500 '€ 2.900,00' 6500 4000 3000 10000 2500 5000
 11500 1 1000 1200 0 1500 '€ 11398,00' 11111 6000]


In [23]:
# Обработка нечисловых значений: удаление '€' и замена запятых на точки
deals['Initial Amount Paid'] = deals['Initial Amount Paid'] \
    .replace(r'[€]', '', regex=True) \
    .replace(r'\s+', '', regex=True) \
    .replace(r'\.', '', regex=True) \
    .replace(r',', '.', regex=True) \
    .astype(float)
deals['Offer Total Amount'] = deals['Offer Total Amount'] \
    .replace(r'[€]', '', regex=True) \
    .replace(r'\s+', '', regex=True) \
    .replace(r'\.', '', regex=True) \
    .replace(r',', '.', regex=True) \
    .astype(float)

In [24]:
# Преобразование типов данных

# Deal Owner Name: object -> category
deals['Deal Owner Name'] = deals['Deal Owner Name'].astype('category')

# Closing Date: object -> datetime (оставили пропуски)
deals['Closing Date'] = pd.to_datetime(deals['Closing Date'], format='%d.%m.%Y', dayfirst=True, errors='coerce')

# Quality: object -> category
deals['Quality'] = deals['Quality'].astype('category')

# Stage: object -> category
deals['Stage'] = deals['Stage'].astype('category')

# Lost Reason: object -> category
deals['Lost Reason'] = deals['Lost Reason'].astype('category')

# Page: object -> category
deals['Page'] = deals['Page'].astype('category')

# Campaign: object -> category
deals['Campaign'] = deals['Campaign'].astype('category')

# Content: object -> category
deals['Content'] = deals['Content'].astype('category')

# Term: object -> category
deals['Term'] = deals['Term'].astype('category')

# Source: object -> category
deals['Source'] = deals['Source'].astype('category')

# Payment Type: object -> category
deals['Payment Type'] = deals['Payment Type'].astype('category')

# Product: object -> category
deals['Product'] = deals['Product'].astype('category')

# Education Type: object -> category
deals['Education Type'] = deals['Education Type'].astype('category')

# Created Time: object -> datetime
deals['Created Time'] = pd.to_datetime(deals['Created Time'], format='%d.%m.%Y %H:%M', dayfirst=True, errors='coerce')

# Course duration: float64 -> Int8
deals['Course duration'] = deals['Course duration'].astype('Int8')

# Months of study: float64 -> Int8
deals['Months of study'] = deals['Months of study'].astype('Int8')

# Initial Amount Paid: object -> Int32
deals['Initial Amount Paid'] = pd.to_numeric(deals['Initial Amount Paid'], errors='coerce').astype('Int32')

# Offer Total Amount: object -> Int32
deals['Offer Total Amount'] = pd.to_numeric(deals['Offer Total Amount'], errors='coerce').astype('Int32')

# Contact Name: float64 -> Int64
#deals['Contact Name'] = deals['Contact Name'].astype('Int64')

# City: object -> category
deals['City'] = deals['City'].astype('category')

# Level of Deutsch: object -> category
deals['Level of Deutsch'] = deals['Level of Deutsch'].astype('category')

Преобразуем значения в столбце SLA в секунды:

определим функцию преобразования time_str_to_seconds.
прменим функцию к каждому значению в столбце SLA.

In [25]:
# Определим функцию для преобразования времени в секунды
def time_str_to_seconds(time_str: str) -> int:
    if isinstance(time_str, int):
        return time_str
    elif isinstance(time_str, float) or pd.isna(time_str):
        return 0
    elif isinstance(time_str, time):
        return time_str.hour * 3600 + time_str.minute * 60 + time_str.second
    elif isinstance(time_str, timedelta):
        return int(time_str.total_seconds())
    else:
        try:
            hours, minutes, seconds = map(int, time_str.split(':'))
            total_seconds = hours * 3600 + minutes * 60 + seconds
            return total_seconds
        except:
            return 0

# Применение функции к столбцу SLA
deals['SLA'] = deals['SLA'].apply(time_str_to_seconds)

In [26]:
# 1. Заменяем "-" на Unknown
deals["City"] = deals["City"].apply(lambda x: "Unknown" if str(x).strip() == "-" else x)

# 2. Убираем вторую часть, если есть запятая
deals["City"] = deals["City"].astype(str).apply(lambda x: x.split(",")[0].strip() if "," in x else x)


# Находим уникальные значения с запятой
values_with_comma = deals.loc[
    deals["City"].astype(str).str.contains(",", na=False),
    "City"
].unique()

# Проходим по всем уникальным значениям с запятой и спрашиваем, что оставить


for val in values_with_comma:
    print(f"\nНайдено значение: {val}")
    new_val = input("Введите, что оставить: ").strip()
    deals.loc[deals["City"] == val, "City"] = new_val

In [27]:
deals["City"] = deals["City"].replace(
    {"Villingen\u2011Schwenningen": "Villingen-Schwenningen"}  # NBHY → обычный дефис
)

In [28]:
# Очистка и приведение столбца уровня немецкого языка 
import re

LEVELS = ["A0","A1","A2","B1","B2","C1","C2"]
RANK = {lvl: i for i, lvl in enumerate(LEVELS)}

# кириллица → латиница для уровней
TRANS = str.maketrans({"а":"a","А":"A","в":"b","В":"B","б":"b","Б":"B","с":"c","С":"C"})

# триггеры высокого уровня
C1_TRIGGERS = ("граждан", "живу", "живем", "живём", "живет", "живёт")

# маркеры языка (для отфильтровки английского)
GER_MARKERS = ("ня", "нем", "de", "deutsch", "german")
ENG_MARKERS = ("ая", "англ", "english")


def normalize_level_cell(x):
    if pd.isna(x) or x == "":
        return "Unknown"
    
    s = str(x).lower()
    
    # Убираем + и - (например, A1+, B2-)
    s = re.sub(r"[\+\-]", "", s)
    
    # Заменяем / на пробел (например, "a1/b1" → "a1 b1")
    s = s.replace("/", " ")
    
    # Убираем лишние пробелы
    s = re.sub(r"\s+", " ", s).strip()
    
    # Транслит кириллицы → латиница (если нужно)
    translit = {
        'а': 'a', 'б': 'b', 'в': 'v', 'г': 'g', 'д': 'd',
        'е': 'e', 'ё': 'e', 'ж': 'zh', 'з': 'z', 'и': 'i',
        'й': 'y', 'к': 'k', 'л': 'l', 'м': 'm', 'н': 'n',
        'о': 'o', 'п': 'p', 'р': 'r', 'с': 's', 'т': 't',
        'у': 'u', 'ф': 'f', 'х': 'h', 'ц': 'ts', 'ч': 'ch',
        'ш': 'sh', 'щ': 'sch', 'ъ': '', 'ы': 'y', 'ь': '',
        'э': 'e', 'ю': 'yu', 'я': 'ya'
    }
    
    s = ''.join(translit.get(c, c) for c in s)
    
    # Приводим к стандартным уровням: A1, A2, B1, B2, C1, C2
    if "a1" in s:
        return "A1"
    elif "a2" in s:
        return "A2"
    elif "b1" in s:
        return "B1"
    elif "b2" in s:
        return "B2"
    elif "c1" in s:
        return "C1"
    elif "c2" in s:
        return "C2"
    else:
        return "Unknown"

# Применяем функцию
deals["Deutsch Level (Normalized)"] = deals["Level of Deutsch"].apply(normalize_level_cell)

In [29]:
# проверка пропущенных значений
deals.isnull().sum()

Id                                2
Deal Owner Name                  31
Closing Date                   6950
Quality                        2255
Stage                             2
Lost Reason                    5471
Page                              2
Campaign                       5528
SLA                               0
Content                        7448
Term                           9141
Source                            2
Payment Type                  21099
Product                       18003
Education Type                18295
Created Time                      2
Course duration               18008
Months of study               20755
Initial Amount Paid           17430
Offer Total Amount            17410
Contact Name                     63
City                              0
Level of Deutsch              20344
Deutsch Level (Normalized)    20344
dtype: int64

### Анализ и предложения по пропущенным значениям:*

Id - удалим пропущенные значения, так как это идентификатор, и он должен быть заполнен для корректного соединения с другими таблицами.

Deal Owner Name - пропущенные значения заполним значением "Unknown".

Closing Date - поскольку могут пропущенные знанения могут быть значимыми, потому что пропущенные значения в этом контексте могут означать, что сделка еще не закрыта или что дата закрытия не была зарегистрирована. Мы их оставим, проеобразовав этот столбец в формат datetime, используя errors ='coerce', чтобы пропуски оставались как 'NaT', что позволит сохранить эту информацию.

Quality - пропущенные значения заполним значением "Unknown".

Stage - пропущенные значения заполним значением "Unknown".

Lost Reason - пропущенные значения заполним значением "Unknown".

Page - пропущенные значения заполним значением "Unknown".

Campaign - пропущенные значения заполним значением "Unknown".

SLA - пропущенные значения заполним значением "Unknown".

Content - пропущенные значения заполним значением "Unknown".

Term - пропущенные значения заполним значением "Unknown".

Source - пропущенные значения заполним значением "Unknown".

Payment Type - пропущенные значения заполним значением "Unknown".

Product - пропущенные значения заполним значением "Unknown".

Course duration - пропущенные значения заполним медианным значением.

Months of study - пропущенные значения заполним медианным значением.

Initial Amount Paid -пропущенные значения заполним медианным значением.

Offer Total Amount - пропущенные значения заполним медианным значением.

Contact Name - удалим пропущенные значения, так как это идентификатор, 
и он должен быть заполнен для корректного соединения с другими таблицами.

City - пропущенные значения заполним значением "Unknown".

Level of Deutsch - пропущенные значения заполним значением "Unknown".

Education Type - пропущенные значения заполним значением "Unknown".

*P.S.

Заполнение значением "Unknown" выбрано для категориальных столбцов, чтобы сохранить структуру данных и избежать пропусков.

Заполнение медианным значением выбрано для числовых столбцов, чтобы избежать смещения данных, вызванного выбросами.


In [30]:
# Сначала преобразуем нужные столбцы в категориальные
category_columns = [
    'Deal Owner Name', 'Quality', 'Stage', 'Lost Reason', 'Page',
    'Campaign', 'Content', 'Term', 'Source', 'Payment Type',
    'Product', 'City', 'Level of Deutsch', 'Education Type'
]

# Шаг 1: Преобразуем в category
for col in category_columns:
    if deals[col].dtype != 'category':
        deals[col] = deals[col].astype('category')

# Шаг 2: Добавляем 'Unknown' как категорию, если её нет, и заполняем пропуски
for col in category_columns:
    if 'Unknown' not in deals[col].cat.categories:
        deals[col] = deals[col].cat.add_categories(['Unknown'])
    deals[col] = deals[col].fillna('Unknown')

In [31]:
# Заполним пропуски медианными значениями для числовых данных
median_columns = ['Course duration', 'Months of study', 'Initial Amount Paid', 'Offer Total Amount']
for col in median_columns:
    deals[col] = deals[col].fillna(deals[col].median())

In [32]:
# Удалим строки с пропусками в идентификаторах
deals = deals.dropna(subset=['Id', 'Contact Name'])

In [33]:
# Проверка данных после заполнения пропусков и преобразования
print(deals.isnull().sum(), "\n")
deals.info()
deals.head()

Id                                0
Deal Owner Name                   0
Closing Date                   6927
Quality                           0
Stage                             0
Lost Reason                       0
Page                              0
Campaign                          0
SLA                               0
Content                           0
Term                              0
Source                            0
Payment Type                      0
Product                           0
Education Type                    0
Created Time                      0
Course duration                   0
Months of study                   0
Initial Amount Paid               0
Offer Total Amount                0
Contact Name                      0
City                              0
Level of Deutsch                  0
Deutsch Level (Normalized)    20295
dtype: int64 

<class 'pandas.core.frame.DataFrame'>
Index: 21532 entries, 0 to 21592
Data columns (total 24 columns):
 #   Column      

,Id,Deal Owner Name,Closing Date,Quality,Stage,Lost Reason,Page,Campaign,SLA,Content,...,Education Type,Created Time,Course duration,Months of study,Initial Amount Paid,Offer Total Amount,Contact Name,City,Level of Deutsch,Deutsch Level (Normalized)
0,5805028000056864695,Ben Hall,NaT,Unknown,New Lead,Unknown,/eng/test,03.07.23women,0,v16,...,Unknown,2024-06-21 15:30:00,11,5,1000,11000,5805028000056849495,nan,Unknown,NaN
1,5805028000056859489,Ulysses Adams,NaT,Unknown,New Lead,Unknown,/at-eng,Unknown,0,Unknown,...,Morning,2024-06-21 15:23:00,6,5,0,2000,5805028000056834471,nan,Unknown,NaN
2,5805028000056832357,Ulysses Adams,2024-06-21,D - Non Target,Lost,Non target,/at-eng,engwien_AT,1603,b1-at,...,Unknown,2024-06-21 14:45:00,11,5,1000,11000,5805028000056854421,nan,Unknown,NaN
3,5805028000056824246,Eva Kent,2024-06-21,E - Non Qualified,Lost,Invalid number,/eng,04.07.23recentlymoved_DE,3604,bloggersvideo14com,...,Unknown,2024-06-21 13:32:00,11,5,1000,11000,5805028000056889351,nan,Unknown,NaN
4,5805028000056873292,Ben Hall,2024-06-21,D - Non Target,Lost,Non target,/eng,discovery_DE,3192,website,...,Unknown,2024-06-21 13:21:00,11,5,1000,11000,5805028000056876176,nan,Unknown,NaN


### 1.4. Анализ датасета Spend

In [34]:
# Загрузим датасет Deals
spend = pd.read_excel('Spend.xlsx')

# Просмотрим первые строки таблицы, чтобы понять структуру данных
spend.head()

,Date,Source,Campaign,Impressions,Spend,Clicks,AdGroup,Ad
0,2023-07-03,Google Ads,gen_analyst_DE,6,0.00,0,NaN,NaN
1,2023-07-03,Google Ads,performancemax_eng_DE,4,0.01,1,NaN,NaN
2,2023-07-03,Facebook Ads,NaN,0,0.00,0,NaN,NaN
3,2023-07-03,Google Ads,NaN,0,0.00,0,NaN,NaN
4,2023-07-03,CRM,NaN,0,0.00,0,NaN,NaN


In [35]:
# Проверка дубликатов (найдено - 917)
spend.duplicated().sum()

np.int64(917)

In [36]:
# Удаление дубликатов (удалено - 917)
spend = spend.drop_duplicates()

In [37]:
# вывод информации о данных и о пропущенных значениях
spend.info()
spend.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
Index: 19862 entries, 0 to 20778
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   Date         19862 non-null  datetime64[ns]
 1   Source       19862 non-null  object        
 2   Campaign     14785 non-null  object        
 3   Impressions  19862 non-null  int64         
 4   Spend        19862 non-null  float64       
 5   Clicks       19862 non-null  int64         
 6   AdGroup      13951 non-null  object        
 7   Ad           13951 non-null  object        
dtypes: datetime64[ns](1), float64(1), int64(2), object(4)
memory usage: 1.4+ MB


Date              0
Source            0
Campaign       5077
Impressions       0
Spend             0
Clicks            0
AdGroup        5911
Ad             5911
dtype: int64

Анализ и предложения по типам данных:
Source: object - преобразовать в категориальный тип данных.
Campaign: object - преобразовать в категориальный тип данных.
Impressions: int64 - преобразовать в Int32.
Clicks: int64 - преобразовать в Int32.

Анализ и предложения по пропущенным значениям:
Campaign - пропущенные значения заполним значением "Unknown".
AdGroup - удалим столбец, поскольку он содержит большое количество пропущенных значений и он не является ключевым для анализа.
Ad - удалим столбец, поскольку он содержит большое количество пропущенных значений и он не является ключевым для анализа.

In [38]:
# Source: object -> category
spend['Source'] = spend['Source'].astype('category')

# Campaign: object -> category
spend['Campaign'] = spend['Campaign'].astype('category')

# Impressions: int64 -> Int32
spend['Impressions'] = spend['Impressions'].astype('Int64')

# Clicks: int64 -> Int32
spend['Clicks'] = spend['Clicks'].astype('Int64')

# Заполненим пропущенные значения для категориальных данных
category_columns = ['Campaign']

for col in category_columns:
    # Проверим наличие категории 'Unknown' перед добавлением
    if 'Unknown' not in spend[col].cat.categories:
        spend[col] = spend[col].cat.add_categories(['Unknown'])
    spend[col] = spend[col].fillna('Unknown')

In [39]:
# Удалим ненужные столбцы
spend = spend.drop(columns=['AdGroup', 'Ad'])

In [40]:
# Проверка данных после заполнения пропусков и преобразования
print(spend.isnull().sum(), "\n")
spend.info()
spend.head()

Date           0
Source         0
Campaign       0
Impressions    0
Spend          0
Clicks         0
dtype: int64 

<class 'pandas.core.frame.DataFrame'>
Index: 19862 entries, 0 to 20778
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   Date         19862 non-null  datetime64[ns]
 1   Source       19862 non-null  category      
 2   Campaign     19862 non-null  category      
 3   Impressions  19862 non-null  Int64         
 4   Spend        19862 non-null  float64       
 5   Clicks       19862 non-null  Int64         
dtypes: Int64(2), category(2), datetime64[ns](1), float64(1)
memory usage: 856.6 KB


,Date,Source,Campaign,Impressions,Spend,Clicks
0,2023-07-03,Google Ads,gen_analyst_DE,6,0.00,0
1,2023-07-03,Google Ads,performancemax_eng_DE,4,0.01,1
2,2023-07-03,Facebook Ads,Unknown,0,0.00,0
3,2023-07-03,Google Ads,Unknown,0,0.00,0
4,2023-07-03,CRM,Unknown,0,0.00,0


### 1.5. Сохранение очищенных датафреймов

In [41]:
import pickle

# Список датафреймов для сохранения
data = [calls, contacts, deals, spend] 

# Путь к файлу
file_path = 'data.pickle'

# Сохранение в pickle-файл
with open(file_path, 'wb') as f:
    pickle.dump(data, f)

print(f"Данные успешно сохранены в {file_path}")

Данные успешно сохранены в data.pickle


In [42]:
# Сохранение каждого датафрейма в отдельный файл
calls.to_excel('calls_clean.xlsx', index=False)
contacts.to_excel('contacts_clean.xlsx', index=False)
deals.to_excel('deals_clean.xlsx', index=False)
spend.to_excel('spend_clean.xlsx', index=False)

print("Все DataFrame сохранены в отдельные Excel-файлы!")

Все DataFrame сохранены в отдельные Excel-файлы!
